# CRYCHIC 单组别常规通讯分析

本教程面向只有一个生物学条件/组别、但可包含多个 sample 和 subject 的常规分析。默认离线小数据会完整走通输入校验、dry-run、LR availability、sender evidence、结果持久化和绘图；也可以通过环境变量换成自己的 H5AD 与 checksum-pinned CellChatDB/CellPhoneDB 资源。

**可解释范围**：单组数据可以描述 LR availability、候选 sender 分配和 receptor 支持；没有组间 contrast，因此 receiver response differential、下游 attribution、整合 `comm_strength`、p/q 值和 communication probability 都不可估计。`not_estimable` 是缺少估计设计，不是生物学零。


In [1]:
%%time
from __future__ import annotations

import hashlib
import os
import shutil
import time
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse

import crychic
from crychic.resources import (
    GeneNamespace,
    Interaction,
    MappingReport,
    ResourceBundle,
    Species,
)
from crychic.sender import SenderEvidenceParameters

SEED = 20260718
NOTEBOOK_STARTED = time.perf_counter()
USE_SYNTHETIC = os.environ.get("CRYCHIC_SINGLE_GROUP_H5AD") is None
INPUT_H5AD = Path(
    os.environ.get("CRYCHIC_SINGLE_GROUP_H5AD", "single_group_input.h5ad")
).expanduser()
DATABASE_ROOT = Path(
    os.environ.get("CRYCHIC_DATABASE_ROOT", "resources")
).expanduser()
LR_RESOURCE = os.environ.get("CRYCHIC_LR_RESOURCE", "cellchat").lower()
SOURCE_CONDITION = os.environ.get("CRYCHIC_SINGLE_GROUP_CONDITION")
OUTPUT_ROOT = Path(
    os.environ.get(
        "CRYCHIC_TUTORIAL_OUTPUT_DIR",
        "tutorial_output/single_group_communication",
    )
).expanduser()
OVERWRITE_OUTPUT = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"CRYCHIC {crychic.__version__}; output={OUTPUT_ROOT.as_posix()}")


CRYCHIC 0.0.1.dev56+g87153a38e.d20260716; output=../../benchmark_work/tutorial_runs/20260719_021000/single_group
CPU times: user 1.5 s, sys: 225 ms, total: 1.72 s
Wall time: 1.72 s


## 1. 输入数据

`adata.layers['counts']` 必须是非负、integer-like 原始计数。`sample_id` 是采样/文库单位，`subject_id` 是独立生物学重复，二者不要因为当前只有一组就混为同一统计角色；`cell_type` 标记 sender/receiver 群体，`condition` 在本教程中只有一个水平。细胞只用于 sample-level 聚合，不能当作独立重复。


In [2]:
%%time
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def make_single_group_adata(n_subjects: int = 4) -> ad.AnnData:
    rng = np.random.default_rng(SEED)
    genes = ("H1", "H2", "L1", "L2", "R1", "R2", "T1", "T2")
    gene_index = {gene: index for index, gene in enumerate(genes)}
    rows: list[np.ndarray] = []
    obs_rows: list[dict[str, str]] = []
    obs_names: list[str] = []
    for subject_index in range(n_subjects):
        subject_id = f"subject-{subject_index + 1:02d}"
        sample_id = f"sample-{subject_index + 1:02d}"
        profiles = {
            "Sender_A": {"L1": 30.0, "L2": 5.0},
            "Sender_B": {"L1": 5.0, "L2": 24.0},
            "Receiver": {
                "R1": 22.0,
                "R2": 18.0,
                "T1": 7.0 + subject_index,
                "T2": 5.0 + subject_index,
            },
        }
        for cell_type, profile in profiles.items():
            for cell_index in range(5):
                means = np.full(len(genes), 2.0, dtype=float)
                means[gene_index["H1"]] = 18.0
                means[gene_index["H2"]] = 14.0
                for gene, value in profile.items():
                    means[gene_index[gene]] = value
                rows.append(rng.poisson(means).astype(np.int32))
                obs_rows.append(
                    {
                        "sample_id": sample_id,
                        "subject_id": subject_id,
                        "cell_type": cell_type,
                        "condition": "baseline",
                    }
                )
                obs_names.append(
                    f"{sample_id}-{cell_type}-{cell_index:02d}"
                )
    counts = sparse.csr_matrix(np.vstack(rows), dtype=np.int32)
    result = ad.AnnData(
        X=sparse.csr_matrix(counts.shape, dtype=np.float64),
        obs=pd.DataFrame(obs_rows, index=obs_names),
        var=pd.DataFrame(index=genes),
    )
    result.layers["counts"] = counts
    return result


if USE_SYNTHETIC:
    adata = make_single_group_adata()
    data_origin = "deterministic_single_group_fixture"
    input_digest = hashlib.sha256(b"crychic-single-group-tutorial-v1").hexdigest()
else:
    if not INPUT_H5AD.is_file():
        raise FileNotFoundError(INPUT_H5AD)
    adata = ad.read_h5ad(INPUT_H5AD)
    required_obs = {"condition", "subject_id", "sample_id", "cell_type"}
    missing_obs = required_obs.difference(adata.obs.columns)
    if missing_obs:
        raise ValueError(f"Missing required obs columns: {sorted(missing_obs)}")
    if "counts" not in adata.layers:
        raise ValueError("adata.layers['counts'] is required")
    available_conditions = tuple(sorted(set(adata.obs["condition"].astype(str))))
    selected_condition = SOURCE_CONDITION
    if selected_condition is None:
        if len(available_conditions) != 1:
            raise ValueError(
                "Set CRYCHIC_SINGLE_GROUP_CONDITION when the input has "
                f"multiple conditions: {available_conditions}"
            )
        selected_condition = available_conditions[0]
    if selected_condition not in available_conditions:
        raise ValueError(
            f"Unknown condition {selected_condition!r}; available={available_conditions}"
        )
    adata = adata[
        adata.obs["condition"].astype(str).eq(selected_condition)
    ].copy()
    adata.obs["condition"] = "single_group"
    data_origin = f"environment_h5ad_condition_subset:{selected_condition}"
    source_digest = sha256_file(INPUT_H5AD)
    input_digest = hashlib.sha256(
        f"{source_digest}|condition={selected_condition}".encode("utf-8")
    ).hexdigest()

TUTORIAL_CONTEXT = str(adata.obs["condition"].astype(str).iloc[0])
TUTORIAL_RECEIVER = str(
    adata.obs["cell_type"].astype(str).value_counts().index[0]
)
print(f"data_origin={data_origin}; shape={adata.shape}")
print(f"context={TUTORIAL_CONTEXT}; receiver={TUTORIAL_RECEIVER}")
adata.obs[["condition", "subject_id", "sample_id", "cell_type"]].value_counts().rename("n_cells")


data_origin=environment_h5ad_condition_subset:CTRL; shape=(3802, 29126)
context=single_group; receiver=Cardiomyocyte
CPU times: user 1.04 s, sys: 168 ms, total: 1.2 s
Wall time: 1.22 s


condition     subject_id  sample_id  cell_type    
single_group  P7          CK357      Cardiomyocyte    541
              P17         CK374      Cardiomyocyte    337
              P8          CK358      Cardiomyocyte    334
              P1          CK158      Cardiomyocyte    328
              P8          CK358      Fibroblast       264
                                                     ... 
              P17         CK358      Pericyte           0
                                     Adipocyte          0
                                     Mast               0
                                     Lymphoid           0
                                     vSMCs              0
Name: n_cells, Length: 176, dtype: int64

In [3]:
%%time
config = crychic.CrychicConfig(
    context_keys=("condition",),
    counts_layer="counts",
    sample_key="sample_id",
    subject_key="subject_id",
    cell_type_key="cell_type",
    design="~ condition",
    random_seed=SEED,
)
validated = crychic.Crychic(config).validate(adata)
validation_summary = {
    "mode": validated.report.mode.value,
    "n_cells": validated.report.n_obs,
    "n_genes": validated.report.n_vars,
    "n_samples": validated.report.n_samples,
    "n_subjects": validated.report.n_subjects,
    "reason_codes": validated.report.reason_codes,
}
validation_summary


CPU times: user 38.6 ms, sys: 10.1 ms, total: 48.7 ms
Wall time: 48 ms


{'mode': 'counts',
 'n_cells': 3802,
 'n_genes': 29126,
 'n_samples': 4,
 'n_subjects': 4,
 'reason_codes': ()}

## 2. 冻结 LR 资源

默认的两个 LR 仅用于让教程离线执行，不是生物学参考数据库。真实分析应加载物种、gene namespace、版本、checksum、license 和 citation 均已审核的完整资源；替换输入时必须同时替换本单元的 synthetic resource。


In [4]:
%%time
def tutorial_interaction(
    interaction_id: str, ligand: str, receptor: str
) -> Interaction:
    return Interaction(
        interaction_id=interaction_id,
        source_interaction_id=interaction_id,
        ligand_name=ligand,
        receptor_name=receptor,
        ligand_subunits=(ligand,),
        receptor_subunits=(receptor,),
        ligand_is_complex=False,
        receptor_is_complex=False,
        direction="Ligand-Receptor",
        source="synthetic_single_group_tutorial",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
        annotation="Synthetic tutorial contract; not a biological reference",
    )


def tutorial_lr_resource() -> ResourceBundle:
    interactions = (
        tutorial_interaction("tutorial_i1", "L1", "R1"),
        tutorial_interaction("tutorial_i2", "L2", "R2"),
    )
    return ResourceBundle(
        resource_id="crychic_single_group_tutorial_lr",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
        interactions=interactions,
        mapping_report=MappingReport(
            source_rows=2, loaded_rows=2, mapped_entities=4
        ),
        manifest_digest=hashlib.sha256(
            b"crychic-single-group-tutorial-lr-v1"
        ).hexdigest(),
        source_files=("embedded_synthetic_single_group_lr",),
        license="CC0-1.0",
        citation="Synthetic CRYCHIC tutorial fixture; not a biological reference.",
    )


if USE_SYNTHETIC:
    lr_resource = tutorial_lr_resource()
    resource_origin = "embedded_synthetic_contract"
elif LR_RESOURCE == "cellchat":
    lr_resource = crychic.load_cellchat_resource(
        DATABASE_ROOT, Species.HUMAN
    )
    resource_origin = "checksum_pinned_cellchat"
elif LR_RESOURCE == "cellphonedb":
    lr_resource = crychic.load_cellphonedb_resource(DATABASE_ROOT)
    resource_origin = "checksum_pinned_cellphonedb"
else:
    raise ValueError("CRYCHIC_LR_RESOURCE must be cellchat or cellphonedb")

print(
    f"resource_origin={resource_origin}; "
    f"interactions={len(lr_resource.interactions)}"
)


resource_origin=checksum_pinned_cellchat; interactions=3233
CPU times: user 136 ms, sys: 11 ms, total: 147 ms
Wall time: 147 ms


## 3. Dry-run 与 availability-only 拟合

先检查资源映射和每个 cell type 的 sample/subject 支持。单一 context 会出现 `single_context_no_response_contrast`，这是预期提示，不会阻止 availability-only 分析。


In [5]:
%%time
model = crychic.Crychic(config, resource_bundle=lr_resource)
plan = model.dry_run(
    adata,
    min_cells=2,
    min_samples_per_context=2,
    min_subjects_per_context=2,
)
stage_plan = pd.DataFrame(
    [
        {
            "stage": stage.name,
            "status": stage.status.value,
            "detail": stage.detail,
        }
        for stage in plan.stages
    ]
)
print(f"can_fit={plan.can_fit}; warnings={plan.warnings}")
stage_plan


can_fit=True; warnings=('single_context_no_response_contrast', 'target_prior_not_provided')
CPU times: user 81.8 ms, sys: 14.1 ms, total: 95.9 ms
Wall time: 94.9 ms


,stage,status,detail
0,validate,ready,AnnData contract validated
1,pseudobulk,ready,sample x cell-type aggregation with min_cells=2
2,context_graph,ready,1 nodes and 0 edges
3,design_audit,ready,formula=~ condition; rank=1/1; registered cont...
4,gene_response,skipped,single context: no response contrast is defined
5,availability,ready,batch LR state and ecosystem availability
6,sender_assignment,ready,context-level non-causal sender evidence from ...
7,attribution,skipped,condition-blind receiver gates and positive Ta...
8,scoring,skipped,one exploratory in-sample functional shared by...


In [6]:
%%time
result_dir = OUTPUT_ROOT / "result"
if OVERWRITE_OUTPUT and result_dir.exists():
    shutil.rmtree(result_dir)

result = model.fit(
    adata,
    output_dir=result_dir,
    input_digest=input_digest,
    persist_edge_evidence=True,
    sender_parameters=SenderEvidenceParameters(
        min_subjects=2 if USE_SYNTHETIC else 1
    ),
    min_cells=2 if USE_SYNTHETIC else 10,
    min_samples_per_context=2,
    min_subjects_per_context=2,
    min_pooled_availability=0.0,
    max_interactions=None,
)
assert isinstance(result, crychic.CrychicResult)
result = crychic.CrychicResult.load(result_dir)
result.manifest["run_id"]


CPU times: user 20min 4s, sys: 18.9 s, total: 20min 23s
Wall time: 20min 23s


'baseline_run_e7f7cc281334c654a8f2492bcc96d76b'

## 4. 查询、审核和绘图

排名函数在 `comm_strength` 全部不可用时自动使用 `availability`，但解释时仍应显式查看 `contrast/status/reason_code`。不要跨 receiver 合并成一个全局榜单。


In [7]:
%%time
interactions = result.read_table("interactions")
sample_scores = result.read_table("sample_scores")
responses = result.read_table("responses")
differential = result.read_table("differential")
stages = {stage["name"]: stage for stage in result.manifest["stages"]}

assert plan.can_fit
assert set(interactions["contrast"].astype(str)) == {"availability_only"}
assert interactions["availability"].notna().all()
assert interactions["comm_strength"].isna().all()
assert interactions["comm_probability"].isna().all()
assert sample_scores["comm_strength"].isna().all()
assert responses.empty
assert stages["response"] == {
    "name": "response",
    "status": "not_estimable",
    "reason_code": "no_estimable_response_contrast",
}
for field in ("p_value", "q_value", "standard_error"):
    if field in differential:
        assert differential[field].isna().all()

status_audit = (
    interactions.groupby(
        ["mode", "status", "reason_code"],
        observed=True,
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
)
status_audit


CPU times: user 1.32 s, sys: 230 ms, total: 1.55 s
Wall time: 1.54 s


,mode,status,reason_code,rows
0,ecosystem,ok,v0_1_inferential_disabled;integrated_strength_...,280800
1,state,ok,v0_1_inferential_disabled;integrated_strength_...,280800


In [8]:
%%time
ranked = result.rank_interactions(
    context={"condition": TUTORIAL_CONTEXT},
    receiver=TUTORIAL_RECEIVER,
    contrast="availability_only",
    mode="state",
    top_n=12,
)
ranked.loc[
    :,
    [
        "sender",
        "receiver",
        "interaction_id",
        "availability",
        "assignment_weight",
        "status",
        "reason_code",
    ],
]


CPU times: user 139 ms, sys: 44 ms, total: 183 ms
Wall time: 182 ms


,sender,receiver,interaction_id,availability,assignment_weight,status,reason_code
0,Cardiomyocyte,Cardiomyocyte,interaction_d69034f60af91d852cf06cbabd1a10e5,0.967827,0.117223,ok,v0_1_inferential_disabled;integrated_strength_...
1,Cardiomyocyte,Cardiomyocyte,interaction_12ff1922985566238442b7bd90d773e4,0.966760,0.112058,ok,v0_1_inferential_disabled;integrated_strength_...
2,Cardiomyocyte,Cardiomyocyte,interaction_ac4538a74a2a7619b75f71325fc5a138,0.955434,0.125188,ok,v0_1_inferential_disabled;integrated_strength_...
3,Endothelial,Cardiomyocyte,interaction_12ff1922985566238442b7bd90d773e4,0.916816,0.109909,ok,v0_1_inferential_disabled;integrated_strength_...
4,Neuronal,Cardiomyocyte,interaction_9d67ab4cfedde5c0a61e95dee959f7d4,0.909274,0.166128,ok,v0_1_inferential_disabled;integrated_strength_...
5,Cardiomyocyte,Cardiomyocyte,interaction_ca22d888573efdb06f61fe50210f7e24,0.904141,0.130007,ok,v0_1_inferential_disabled;integrated_strength_...
6,Neuronal,Cardiomyocyte,interaction_0b3f93b3dca82f472f09cff5d109ce7b,0.854962,0.165993,ok,v0_1_inferential_disabled;integrated_strength_...
7,Neuronal,Cardiomyocyte,interaction_61ae3cdbb14adc4eae30296c7837d601,0.771482,0.166128,ok,v0_1_inferential_disabled;integrated_strength_...
8,Cardiomyocyte,Cardiomyocyte,interaction_97d16faa17726bfb2fdd6462c01a42d2,0.746272,0.103874,ok,v0_1_inferential_disabled;integrated_strength_...
9,vSMCs,Cardiomyocyte,interaction_12ff1922985566238442b7bd90d773e4,0.745229,0.102825,ok,v0_1_inferential_disabled;integrated_strength_...


In [9]:
%%time
plot_data = ranked.loc[ranked["availability"].notna()].copy()
if plot_data.empty:
    print("No estimable availability rows; inspect status_audit.")
else:
    plot_data = plot_data.sort_values("availability", kind="stable")
    plot_data["edge"] = (
        plot_data["sender"].astype(str)
        + " -> "
        + plot_data["receiver"].astype(str)
        + " | "
        + plot_data["interaction_id"].astype(str)
    )
    fig, ax = plt.subplots(
        figsize=(7.2, max(3.0, 0.42 * len(plot_data)))
    )
    colors = np.where(
        plot_data["sender"].astype(str).eq(TUTORIAL_RECEIVER),
        "#2E6F95",
        "#C75B39",
    )
    ax.barh(plot_data["edge"], plot_data["availability"], color=colors)
    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel("LR availability (descriptive)")
    ax.set_ylabel("")
    ax.set_title("Single-group availability-only ranking")
    fig.tight_layout()
    figure_path = OUTPUT_ROOT / "single_group_availability.png"
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"figure={figure_path}")
tutorial_elapsed_seconds = time.perf_counter() - NOTEBOOK_STARTED
print(f"tutorial_elapsed_seconds={tutorial_elapsed_seconds:.1f}")


figure=../../benchmark_work/tutorial_runs/20260719_021000/single_group/single_group_availability.png
tutorial_elapsed_seconds=1227.1
CPU times: user 282 ms, sys: 38 ms, total: 320 ms
Wall time: 330 ms


## 结果边界与真实数据替换

- 将 `CRYCHIC_SINGLE_GROUP_H5AD` 指向自己的 H5AD；不要把 normalized/log expression 重命名为 `counts`。
- 将 `CRYCHIC_DATABASE_ROOT` 指向 checksum-pinned 数据库，并用 `CRYCHIC_LR_RESOURCE=cellchat` 或 `cellphonedb` 选择资源。
- 单组分析的 `availability_only` 适合质量控制、候选通讯描述和后续实验假设生成；它不能验证组间变化，也不能借助 target prior 凭空产生 differential response。
- 若有两个或更多条件及足够独立 subject，请使用下一份多组教程，预先声明 contrast 并运行 subject-blocked cross-fit。
